In [ ]:
import gc
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
import torch.nn.functional as F
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm

sys.path.insert(0, str(Path('../src').resolve()))

from audio import get_audio_duration, is_inat_file, load_and_chunk
from features import (
    build_inat_mapping,
    compute_sample_weight,
    cyclic_encode,
    make_label_vector,
    make_soundscape_label,
    parse_soundscape_hour,
 )

CONFIG = {
    'base_dir': Path('../Data'),
    'out_dir': Path('../precomputed'),
    'perch_path': 'https://kaggle.com/models/google/bird-vocalization-classifier/TensorFlow2/bird-vocalization-classifier/4',
    'sr': 32_000,
    'chunk_dur': 5.0,
    'chunk_samp': 160_000,
    'perch_batch': 64,
    'warmup_passes': 5,
    'n_classes': 234,
    'secondary_soft': 0.3,
    'n_folds': 5,
    'random_state': 42,
    'gamma': -0.5,
    'rarity_clip': 10.0,
    'held_out_count': 10,
    'weight_inat': 0.5,
    'weight_xc_high': 1.0,
    'weight_xc_med': 0.8,
    'weight_xc_low': 0.6,
    'weight_unrated': 0.5,
}

def build_output_dirs(base_out: Path) -> None:
    dirs = [
        base_out,
        base_out / 'metadata',
        base_out / 'label_vectors',
        base_out / 'perch_embeddings',
        base_out / 'perch_embeddings' / 'train_audio',
        base_out / 'perch_embeddings' / 'soundscapes',
    ]
    for d in dirs:
        d.mkdir(parents=True, exist_ok=True)

def get_embedding_path(out_dir: Path, species_id: str, audio_filename: str) -> Path:
    stem = Path(audio_filename).stem
    return out_dir / 'perch_embeddings' / 'train_audio' / species_id / f'{stem}.npy'

build_output_dirs(CONFIG['out_dir'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch ready')
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', device)

PyTorch ready
Torch: 2.6.0+cu124
CUDA available: True
Device: cuda


In [ ]:
train_df = pd.read_csv(CONFIG['base_dir'] / 'train.csv')
tax_df = pd.read_csv(CONFIG['base_dir'] / 'taxonomy.csv')
sc_df = pd.read_csv(CONFIG['base_dir'] / 'train_soundscapes_labels.csv').drop_duplicates().reset_index(drop=True)

inat_to_primary = build_inat_mapping(tax_df)

le = LabelEncoder()
le.fit(tax_df['primary_label'].astype(str).values)
species_to_class = dict(zip(tax_df['primary_label'].astype(str), tax_df['class_name'].astype(str)))

train_df['filename'] = train_df['filename'].astype(str)
train_df['audio_path'] = train_df['filename'].apply(lambda x: CONFIG['base_dir'] / 'train_audio' / x)

sample_paths = train_df['audio_path'].sample(n=min(5, len(train_df)), random_state=42).tolist()
for p in sample_paths:
    print(p, p.exists())

print('n species:', len(le.classes_))
print('n clips:', len(train_df))
print('n SC labels:', len(sc_df))

..\Data\train_audio\purjay1\XC299441.ogg True
..\Data\train_audio\saytan1\iNat588079.ogg True
..\Data\train_audio\thlwre1\XC350961.ogg True
..\Data\train_audio\whbwar2\XC835399.ogg True
..\Data\train_audio\whbwar2\iNat502152.ogg True
n species: 234
n clips: 35549
n SC labels: 739


In [7]:
durations = []
n_chunks = []
is_corrupt = []

for p in tqdm(train_df['audio_path'].tolist(), desc='Duration scan'):
    try:
        dur = float(get_audio_duration(str(p)))
        bad = dur < 0.5
    except Exception:
        dur = 0.0
        bad = True
    durations.append(dur)
    n_chunks.append(int(np.ceil(max(dur, 0.0) / CONFIG['chunk_dur'])) if dur > 0 else 0)
    is_corrupt.append(bool(bad))

train_df['duration'] = durations
train_df['n_chunks'] = n_chunks
train_df['is_corrupt'] = is_corrupt

corrupted_df = train_df[train_df['is_corrupt']].copy()
corrupted_df[['filename', 'audio_path', 'duration']].to_csv(CONFIG['out_dir'] / 'metadata' / 'corrupted_files.csv', index=False)
print('n corrupted:', len(corrupted_df))
print(corrupted_df['filename'].head())

clean_df = train_df[~train_df['is_corrupt']].copy().reset_index(drop=True)

Duration scan:   0%|          | 0/35549 [00:00<?, ?it/s]

n corrupted: 363
8      1161364/iNat1264238.ogg
29      209233/iNat1545859.ogg
127       22973/iNat781822.ogg
128      22973/iNat1353836.ogg
135      22973/iNat1460521.ogg
Name: filename, dtype: str


In [8]:
clean_df['class_name'] = clean_df['primary_label'].astype(str).map(species_to_class).fillna('Unknown')
clean_df['latitude_round'] = pd.to_numeric(clean_df['latitude'], errors='coerce').round(1).fillna(-999.0)
clean_df['author'] = clean_df['author'].astype(str).fillna('unknown')
clean_df['group_key'] = clean_df['author'] + '_' + clean_df['latitude_round'].astype(str)

counts = clean_df['primary_label'].astype(str).value_counts()
few_species = set(counts[counts < 10].index.tolist())
clean_df['fold'] = -1

common_mask = ~clean_df['primary_label'].astype(str).isin(few_species)
sgkf = StratifiedGroupKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=CONFIG['random_state'])
split_df = clean_df[common_mask].copy()

for fold_id, (_, val_idx) in enumerate(sgkf.split(split_df, y=split_df['class_name'], groups=split_df['group_key'])):
    clean_df.loc[split_df.iloc[val_idx].index, 'fold'] = fold_id

print(clean_df['fold'].value_counts(dropna=False).sort_index())
print(clean_df.groupby('fold')['class_name'].nunique())

fold
-1      98
 0    7018
 1    7018
 2    7017
 3    7018
 4    7017
Name: count, dtype: int64
fold
-1    5
 0    4
 1    4
 2    4
 3    4
 4    4
Name: class_name, dtype: int64


In [ ]:
train_labels = np.zeros((len(clean_df), CONFIG['n_classes']), dtype=np.float32)
for i, row in tqdm(clean_df.iterrows(), total=len(clean_df), desc='Label vectors'):
    train_labels[i] = make_label_vector(
        primary_label=str(row['primary_label']),
        secondary_labels=str(row.get('secondary_labels', '')),
        le=le,
        n_classes=CONFIG['n_classes'],
        secondary_soft=CONFIG['secondary_soft'],
        inat_to_primary=inat_to_primary,
    )

np.save(CONFIG['out_dir'] / 'label_vectors' / 'train_labels.npy', train_labels)
species_index = {cls: int(i) for i, cls in enumerate(le.classes_.tolist())}
with open(CONFIG['out_dir'] / 'metadata' / 'species_index.json', 'w', encoding='utf-8') as f:
    json.dump(species_index, f, indent=2)

print('train_labels shape:', train_labels.shape)


Label vectors:   0%|          | 0/35186 [00:00<?, ?it/s]

train_labels shape: (35186, 234)


In [10]:
species_counts = clean_df['primary_label'].astype(str).value_counts()
total_count = len(clean_df)

weights = []
for _, row in tqdm(clean_df.iterrows(), total=len(clean_df), desc='Sample weights'):
    pl = str(row['primary_label'])
    rating = pd.to_numeric(row.get('rating', np.nan), errors='coerce')
    w = compute_sample_weight(
        rating=rating,
        is_inat=is_inat_file(str(row['filename'])),
        species_count=int(species_counts.get(pl, 1)),
        total_count=total_count,
        gamma=CONFIG['gamma'],
        rarity_clip=CONFIG['rarity_clip'],
    )
    weights.append(float(w))

weights = np.asarray(weights, dtype=np.float32)
weights = (weights - weights.min()) / max((weights.max() - weights.min()), 1e-8)
clean_df['sample_weight'] = weights

one_shot_species = set(species_counts[species_counts == 1].index.astype(str).tolist())
few_shot_species = set(species_counts[species_counts < 10].index.astype(str).tolist())
zero_shot_species = {f'47158son{i:02d}' for i in range(1, 26)}
clean_df['is_zero_shot'] = clean_df['primary_label'].astype(str).isin(zero_shot_species)
clean_df['is_one_shot'] = clean_df['primary_label'].astype(str).isin(one_shot_species)
clean_df['is_few_shot'] = clean_df['primary_label'].astype(str).isin(few_shot_species)
clean_df['learning_strategy'] = np.where(clean_df['is_zero_shot'], 'zero_shot', np.where(clean_df['is_one_shot'], 'one_shot', np.where(clean_df['is_few_shot'], 'few_shot', 'normal')))

clean_df.to_csv(CONFIG['out_dir'] / 'metadata' / 'train_folds.csv', index=False)
np.save(CONFIG['out_dir'] / 'metadata' / 'sample_weights.npy', weights.astype(np.float32))

Sample weights:   0%|          | 0/35186 [00:00<?, ?it/s]

In [ ]:
def to_seconds(t):
    hh, mm, ss = [int(v) for v in str(t).split(':')]
    return hh * 3600 + mm * 60 + ss

labeled_files = sorted(sc_df['filename'].astype(str).unique().tolist())
held_out = set(labeled_files[-CONFIG['held_out_count']:])
labeled_train = set(labeled_files[:-CONFIG['held_out_count']])

all_sc_files = sorted([p.name for p in (CONFIG['base_dir'] / 'train_soundscapes').glob('*.ogg')])
pseudo_pool = [x for x in all_sc_files if x not in set(labeled_files)]

rows = []
soundscape_labels = np.zeros((len(sc_df), CONFIG['n_classes']), dtype=np.float32)
for i, row in tqdm(sc_df.iterrows(), total=len(sc_df), desc='Soundscape index'):
    fn = str(row['filename'])
    hour, minute = parse_soundscape_hour(fn)
    hour_sin, hour_cos = cyclic_encode(hour + minute / 60.0, 24.0)
    labels = str(row.get('primary_label', ''))
    species_list = [s.strip() for s in labels.split(';') if s.strip()]
    split = 'held_out' if fn in held_out else 'labeled_train'

    soundscape_labels[i] = make_soundscape_label(
        labels,
        le=le,
        n_classes=CONFIG['n_classes'],
        inat_to_primary=inat_to_primary,
    )
    rows.append({
        'filename': fn,
        'start': row.get('start', '00:00:00'),
        'end': row.get('end', '00:00:05'),
        'start_sec': to_seconds(row.get('start', '00:00:00')),
        'end_sec': to_seconds(row.get('end', '00:00:05')),
        'primary_label': labels,
        'n_species': len(species_list),
        'hour': hour,
        'minute': minute,
        'hour_sin': hour_sin,
        'hour_cos': hour_cos,
        'is_frog_time': bool(hour >= 18 or hour < 6),
        'split': split,
    })

soundscape_windows_df = pd.DataFrame(rows)
soundscape_windows_df.to_csv(CONFIG['out_dir'] / 'metadata' / 'soundscape_windows.csv', index=False)
pd.DataFrame({'filename': pseudo_pool, 'split': 'pseudo_pool'}).to_csv(CONFIG['out_dir'] / 'metadata' / 'pseudo_pool_index.csv', index=False)
np.save(CONFIG['out_dir'] / 'label_vectors' / 'soundscape_labels.npy', soundscape_labels)
print('held_out:', len(held_out), 'labeled_train:', len(labeled_train), 'pseudo_pool:', len(pseudo_pool))


Soundscape index:   0%|          | 0/739 [00:00<?, ?it/s]

held_out: 10 labeled_train: 56 pseudo_pool: 10592


In [27]:
class TorchPerchLike:
    def __init__(self, device: torch.device, source_sr: int = 32000):
        self.device = device
        self.source_sr = source_sr
        self.bundle = torchaudio.pipelines.WAV2VEC2_BASE
        self.target_sr = self.bundle.sample_rate
        self.model = self.bundle.get_model().to(self.device).eval()

    @torch.inference_mode()
    def infer(self, batch_np: np.ndarray) -> np.ndarray:
        x = torch.from_numpy(batch_np).to(self.device)
        if self.source_sr != self.target_sr:
            x = torchaudio.functional.resample(x, self.source_sr, self.target_sr)

        feats, _ = self.model.extract_features(x)
        emb = feats[-1].mean(dim=1)

        if emb.shape[1] < 1280:
            emb = F.pad(emb, (0, 1280 - emb.shape[1]))
        elif emb.shape[1] > 1280:
            emb = emb[:, :1280]

        return emb.detach().cpu().numpy().astype(np.float32, copy=False)

def flush_perch_batch(perch_model, chunk_buffer, perch_batch_size=64):
    n_real = len(chunk_buffer)
    if n_real == 0:
        return np.empty((0, 1280), dtype=np.float32)

    batch = np.zeros((perch_batch_size, CONFIG['chunk_samp']), dtype=np.float32)
    for i, chunk in enumerate(chunk_buffer):
        batch[i] = chunk

    emb = perch_model.infer(batch)
    return emb[:n_real]

def warmup_perch(perch_model, perch_batch_size=64, chunk_samp=160000, n_passes=5):
    print('Warmup passes...')
    dummy = np.zeros((perch_batch_size, chunk_samp), dtype=np.float32)
    for i in range(n_passes):
        _ = perch_model.infer(dummy)
        print(f'  Warmup {i+1}/{n_passes}')

print('Loading PyTorch acoustic encoder...')
perch = TorchPerchLike(device=device, source_sr=CONFIG['sr'])
warmup_perch(
    perch_model=perch,
    perch_batch_size=CONFIG['perch_batch'],
    chunk_samp=CONFIG['chunk_samp'],
    n_passes=CONFIG['warmup_passes'],
)

test_batch = np.zeros((CONFIG['perch_batch'], CONFIG['chunk_samp']), dtype=np.float32)
emb = perch.infer(test_batch)
print('embedding shape:', emb.shape)

Loading PyTorch acoustic encoder...
Warmup passes...
  Warmup 1/5
  Warmup 2/5
  Warmup 3/5
  Warmup 4/5
  Warmup 5/5
embedding shape: (64, 1280)


In [16]:
# Cell 9 — FAST embedding extraction (PyTorch, cross-file batching)
import gc
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

PERCH_BATCH = CONFIG['perch_batch']  # 64
CHUNK_SAMP = CONFIG['chunk_samp']     # 160000

# Optional warmup (keeps timing stable)
print('Warmup passes...')
dummy = np.zeros((PERCH_BATCH, CHUNK_SAMP), dtype=np.float32)
for i in range(CONFIG['warmup_passes']):
    _ = perch.infer(dummy)
    print(f'  Warmup {i+1}/{CONFIG["warmup_passes"]}')
del dummy
gc.collect()
print('Warmup done.\n')

to_process = []
already_done = []

for i, row in clean_df.iterrows():
    rel = Path(str(row['filename']))
    species_id = rel.parts[0]
    audio_name = rel.name
    emb_path = get_embedding_path(CONFIG['out_dir'], species_id, audio_name)

    if emb_path.exists():
        try:
            payload = np.load(emb_path, allow_pickle=True).item()
            nc = int(payload['embeddings'].shape[0])
            already_done.append({
                'filename': str(rel),
                'species_id': species_id,
                'embedding_path': emb_path.as_posix(),
                'fold': int(row.get('fold', -1)),
                'n_chunks': nc,
            })
            continue
        except Exception:
            pass

    audio_path = CONFIG['base_dir'] / 'train_audio' / rel
    to_process.append({
        'row_idx': i,
        'row': row,
        'audio_path': audio_path,
        'emb_path': emb_path,
        'rel': rel,
        'species_id': species_id,
    })

print(f'Already done: {len(already_done):,}')
print(f'To process:   {len(to_process):,}')
print(f'Total:        {len(clean_df):,}\n')

chunk_buffer = []
meta_buffer = []
file_acc = {}
file_nc = {}
train_emb_rows = list(already_done)


def file_key(file_dict):
    return str(file_dict['emb_path'])


def flush_buffer(chunk_buf, meta_buf):
    embs = flush_perch_batch(perch, chunk_buf, perch_batch_size=PERCH_BATCH)
    for emb, (fd, ci) in zip(embs, meta_buf):
        fk2 = file_key(fd)
        file_acc.setdefault(fk2, []).append((ci, emb.copy()))


def save_file(fdict, emb_list):
    emb_path = fdict['emb_path']
    emb_path.parent.mkdir(parents=True, exist_ok=True)

    emb_list = sorted(emb_list, key=lambda x: x[0])
    embs_arr = np.stack([e for _, e in emb_list]).astype(np.float32)

    row = fdict['row']
    row_idx = fdict['row_idx']

    payload = {
        'embeddings': embs_arr,
        'label': train_labels[row_idx].astype(np.float32),
        'weight': float(row.get('sample_weight', 1.0)),
        'fold': int(row.get('fold', -1)),
    }
    np.save(emb_path, payload, allow_pickle=True)

    return {
        'filename': str(fdict['rel']),
        'species_id': fdict['species_id'],
        'embedding_path': emb_path.as_posix(),
        'fold': int(row.get('fold', -1)),
        'n_chunks': int(embs_arr.shape[0]),
    }


for fdict in tqdm(to_process, desc='Perch FAST (PyTorch)'):
    chunks = load_and_chunk(
        str(fdict['audio_path']),
        sr=CONFIG['sr'],
        chunk_samp=CHUNK_SAMP,
    )
    if not chunks:
        continue

    fk = file_key(fdict)
    file_nc[fk] = len(chunks)
    file_acc[fk] = []

    for cidx, chunk in enumerate(chunks):
        chunk_buffer.append(chunk)
        meta_buffer.append((fdict, cidx))

        if len(chunk_buffer) == PERCH_BATCH:
            flush_buffer(chunk_buffer, meta_buffer)
            chunk_buffer.clear()
            meta_buffer.clear()
            gc.collect()

    fk = file_key(fdict)
    if fk in file_acc and len(file_acc[fk]) >= file_nc.get(fk, 9999):
        row_rec = save_file(fdict, file_acc.pop(fk))
        train_emb_rows.append(row_rec)

if chunk_buffer:
    flush_buffer(chunk_buffer, meta_buffer)
    chunk_buffer.clear()
    meta_buffer.clear()
    gc.collect()

for fk, emb_list in file_acc.items():
    fdict = next(f for f in to_process if file_key(f) == fk)
    row_rec = save_file(fdict, emb_list)
    train_emb_rows.append(row_rec)

file_acc.clear()
gc.collect()

train_emb_index_df = pd.DataFrame(train_emb_rows)
train_emb_index_df.to_csv(
    CONFIG['out_dir'] / 'metadata' / 'train_emb_index.csv',
    index=False,
 )

total_chunks = int(train_emb_index_df['n_chunks'].clip(lower=0).sum())
print('\nDone!')
print(f'Files processed: {len(train_emb_index_df):,}')
print(f'Total chunks:    {total_chunks:,}')
print('Saved: metadata/train_emb_index.csv')

Warmup passes...
  Warmup 1/5
  Warmup 2/5
  Warmup 3/5
  Warmup 4/5
  Warmup 5/5
Warmup done.

Already done: 5
To process:   35,181
Total:        35,186



Perch FAST (PyTorch):   0%|          | 0/35181 [00:00<?, ?it/s]


Done!
Files processed: 35,186
Total chunks:    265,561
Saved: metadata/train_emb_index.csv


In [29]:
# Cell 10 — FAST soundscape embeddings (cross-file batching)
import gc
from tqdm.auto import tqdm

PERCH_BATCH = CONFIG['perch_batch']
CHUNK_SAMP = CONFIG['chunk_samp']

split_map = soundscape_windows_df.groupby('filename')['split'].first().to_dict()
all_sc_paths = sorted((CONFIG['base_dir'] / 'train_soundscapes').glob('*.ogg'))

# Resume support
to_process = []
already_done = []
for sc_path in all_sc_paths:
    emb_path = CONFIG['out_dir'] / 'perch_embeddings' / 'soundscapes' / f'{sc_path.stem}.npy'
    split = split_map.get(sc_path.name, 'pseudo_pool')

    if emb_path.exists():
        try:
            payload = np.load(emb_path, allow_pickle=True).item()
            n_chunks_done = int(payload['embeddings'].shape[0])
        except Exception:
            n_chunks_done = -1
        already_done.append({
            'filename': sc_path.name,
            'embedding_path': emb_path.as_posix(),
            'split': split,
            'n_chunks': n_chunks_done,
        })
        continue

    to_process.append({
        'sc_path': sc_path,
        'emb_path': emb_path,
        'split': split,
    })

print(f'Already done: {len(already_done):,}')
print(f'To process:   {len(to_process):,}')

# Cross-file rolling buffers
chunk_buffer = []
meta_buffer = []         # (fdict, chunk_idx)
file_acc = {}            # filename -> [(chunk_idx, emb), ...]
file_nc = {}             # filename -> expected chunk count
soundscape_emb_rows = list(already_done)


def flush_sc_buffer(chunk_buf, meta_buf):
    embs = flush_perch_batch(perch, chunk_buf, perch_batch_size=PERCH_BATCH)
    for emb, (fd, ci) in zip(embs, meta_buf):
        fname = fd['sc_path'].name
        file_acc.setdefault(fname, []).append((ci, emb.copy()))


def save_sc_file(fdict, emb_list):
    emb_list = sorted(emb_list, key=lambda x: x[0])
    embs_arr = np.stack([e for _, e in emb_list]).astype(np.float32)
    starts = np.arange(embs_arr.shape[0], dtype=np.float32) * float(CONFIG['chunk_dur'])

    emb_path = fdict['emb_path']
    emb_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(
        emb_path,
        {
            'embeddings': embs_arr,
            'starts': starts,
            'filename': fdict['sc_path'].name,
        },
        allow_pickle=True,
    )

    return {
        'filename': fdict['sc_path'].name,
        'embedding_path': emb_path.as_posix(),
        'split': fdict['split'],
        'n_chunks': int(embs_arr.shape[0]),
    }


for fdict in tqdm(to_process, desc='Perch soundscapes FAST'):
    chunks = load_and_chunk(str(fdict['sc_path']), sr=CONFIG['sr'], chunk_samp=CHUNK_SAMP)
    if not chunks:
        continue

    fname = fdict['sc_path'].name
    file_nc[fname] = len(chunks)
    file_acc[fname] = []

    for cidx, chunk in enumerate(chunks):
        chunk_buffer.append(chunk)
        meta_buffer.append((fdict, cidx))

        if len(chunk_buffer) == PERCH_BATCH:
            flush_sc_buffer(chunk_buffer, meta_buffer)
            chunk_buffer.clear()
            meta_buffer.clear()

    # Save file immediately once complete
    if fname in file_acc and len(file_acc[fname]) >= file_nc.get(fname, 9999):
        row_rec = save_sc_file(fdict, file_acc.pop(fname))
        soundscape_emb_rows.append(row_rec)

# Flush remaining partial batch once
if chunk_buffer:
    flush_sc_buffer(chunk_buffer, meta_buffer)
    chunk_buffer.clear()
    meta_buffer.clear()

# Save any remaining accumulated files
for fname, emb_list in file_acc.items():
    fdict = next(f for f in to_process if f['sc_path'].name == fname)
    row_rec = save_sc_file(fdict, emb_list)
    soundscape_emb_rows.append(row_rec)

file_acc.clear()
gc.collect()

soundscape_emb_index_df = pd.DataFrame(soundscape_emb_rows)
soundscape_emb_index_df.to_csv(CONFIG['out_dir'] / 'metadata' / 'soundscape_emb_index.csv', index=False)
print('\nDone!')
print(soundscape_emb_index_df['split'].value_counts(dropna=False))

Already done: 207
To process:   10,451


Perch soundscapes FAST:   0%|          | 0/10451 [00:00<?, ?it/s]


Done!
split
pseudo_pool      10592
labeled_train       56
held_out            10
Name: count, dtype: int64


In [30]:
required = [
    CONFIG['out_dir'] / 'metadata' / 'train_folds.csv',
    CONFIG['out_dir'] / 'metadata' / 'sample_weights.npy',
    CONFIG['out_dir'] / 'metadata' / 'soundscape_windows.csv',
    CONFIG['out_dir'] / 'metadata' / 'pseudo_pool_index.csv',
    CONFIG['out_dir'] / 'metadata' / 'corrupted_files.csv',
    CONFIG['out_dir'] / 'metadata' / 'train_emb_index.csv',
    CONFIG['out_dir'] / 'metadata' / 'soundscape_emb_index.csv',
    CONFIG['out_dir'] / 'metadata' / 'species_index.json',
    CONFIG['out_dir'] / 'label_vectors' / 'train_labels.npy',
    CONFIG['out_dir'] / 'label_vectors' / 'soundscape_labels.npy',
]

for p in required:
    status = 'OK' if p.exists() else 'MISSING'
    size_mb = (p.stat().st_size / (1024 ** 2)) if p.exists() else 0.0
    print(f'{p.as_posix()} | {status} | {size_mb:.2f} MB')

if (CONFIG['out_dir'] / 'metadata' / 'train_emb_index.csv').exists():
    tei = pd.read_csv(CONFIG['out_dir'] / 'metadata' / 'train_emb_index.csv')
    print('train chunks total:', int(tei['n_chunks'].sum()) if len(tei) else 0)

if (CONFIG['out_dir'] / 'metadata' / 'soundscape_emb_index.csv').exists():
    sei = pd.read_csv(CONFIG['out_dir'] / 'metadata' / 'soundscape_emb_index.csv')
    print('soundscape chunks total:', int(sei['n_chunks'].sum()) if len(sei) else 0)

print('Upload precomputed/ folder to Kaggle as private dataset')
print('kaggle datasets init -p ./precomputed')
print('kaggle datasets create -p ./precomputed --dir-mode zip')
print('kaggle datasets version -p ./precomputed -m "Added Perch embeddings round 1"')

../precomputed/metadata/train_folds.csv | OK | 10.58 MB
../precomputed/metadata/sample_weights.npy | OK | 0.13 MB
../precomputed/metadata/soundscape_windows.csv | OK | 0.11 MB
../precomputed/metadata/pseudo_pool_index.csv | OK | 0.56 MB
../precomputed/metadata/corrupted_files.csv | OK | 0.03 MB
../precomputed/metadata/train_emb_index.csv | OK | 3.35 MB
../precomputed/metadata/soundscape_emb_index.csv | OK | 1.47 MB
../precomputed/metadata/species_index.json | OK | 0.00 MB
../precomputed/label_vectors/train_labels.npy | OK | 31.41 MB
../precomputed/label_vectors/soundscape_labels.npy | OK | 0.66 MB
train chunks total: 265561
soundscape chunks total: 127896
Upload precomputed/ folder to Kaggle as private dataset
kaggle datasets init -p ./precomputed
kaggle datasets create -p ./precomputed --dir-mode zip
kaggle datasets version -p ./precomputed -m "Added Perch embeddings round 1"
